In [13]:
import ollama  

#all_models = [m.model for m in ollama.list().models]


#print(f"These are the available models: {(all_models)}")

## Deploy Ollama Models 

In [ ]:

SYSTEM_PROMPT = """Du bist ein neutraler politischer Analyst und fasst Reden aus dem Deutschen Bundestag zusammen.
Regeln:
- Antworte ausschließlich auf Deutsch.
- Verwende nur Informationen, die im Text stehen. Erfinde nichts und bewerte nicht.
- Zwischenrufe, Beifall und Begrüßungsformeln ignorierst du."""





CHUNK_PROMPT = """Hier ist Abschnitt {i} von {n} einer Bundestagsrede von {name}.
Fasse die wichtigsten Aussagen, Forderungen und Argumente dieses Abschnitts in knappen Stichpunkten zusammen.

Abschnitt:
\"\"\"
{text}
\"\"\""""

FINAL_PROMPT = """Fasse die folgende Bundestagsrede von {name} zusammen.
Nutze genau diese Struktur in Markdown:

**Thema:** 
- ein Satz

**Kernaussagen:**
- Maximal 3 Stichpunkte

**Forderungen / Positionen:**
- Stichpunkte (oder "keine konkreten Forderungen")

**Kritik an anderen Parteien / der Regierung:**
- Stichpunkte (oder "keine")

{kind}:
\"\"\"
{text}
\"\"\""""





used_model = "llama3:latest"

def ollama_chat(prompt, model):
    stream = ollama.chat(
        model=model,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        stream=True,
    )

    output = ""
    for chunk in stream:
        output += chunk.message.content
        #print(chunk.message.content, end="", flush=True)
    #print("\n")
    return output

output = ollama_chat(prompt, used_model)
print(f"Final output of {used_model} :  {output}")

Hier ist ein kurzer Test in 2 Sätzen:

Gute Bildung ist die Grundlage für ein gutes Leben und entscheidet über Chancen, Teilhabe und Selbstbestimmung. Wir streben an, dass alle Kinder und Jugendlichen ihre Potenziale entfalten und einen anerkannten Schulabschluss erreichen können, unabhängig von Herkunft, Wohnort oder finanziellen Möglichkeiten.

Final output of llama3:latest :  Hier ist ein kurzer Test in 2 Sätzen:

Gute Bildung ist die Grundlage für ein gutes Leben und entscheidet über Chancen, Teilhabe und Selbstbestimmung. Wir streben an, dass alle Kinder und Jugendlichen ihre Potenziale entfalten und einen anerkannten Schulabschluss erreichen können, unabhängig von Herkunft, Wohnort oder finanziellen Möglichkeiten.


## Use Anthropic model (just Haiku, because it is cheap :P )

In [8]:
import os
from dotenv import load_dotenv

print(os.getcwd())          # in welchem Ordner läuft das Notebook?
print(load_dotenv())        # True = .env gefunden, False = nicht gefunden
print(os.getenv("ANTHROPIC_API_KEY", "")[:10])  # sollte "sk-ant-api" zeigen

c:\Users\brand\OneDrive\Desktop\AI_Selbststudium\ML_little\bundestag_reden
True
sk-ant-usr


In [8]:
# from dotenv import load_dotenv
# import anthropic

# load_dotenv()
# client = anthropic.Anthropic()
# response = client.messages.create(
#     model="claude-haiku-4-5-20251001",
#     max_tokens=100,
#     messages=[{"role": "user", "content": "Sag Hallo"}],
# )
# print(response.content[0].text)

### DPI Bundestag Agent 

In [10]:
import os, re, httpx
from pydantic_ai import Agent, ModelRetry
from dotenv import load_dotenv
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider


load_dotenv()  # vor allem anderen, was die Keys braucht

DIP = "https://search.dip.bundestag.de/api/v1"
KEY = os.environ["DIP_API_KEY"]

model = OpenAIChatModel(
    "qwen2.5:7b",
    provider=OpenAIProvider(base_url="http://localhost:11434/v1", api_key="ollama"),
)
agent = Agent(model)

@agent.tool_plain
def suche_vorgaenge(titel: str, datum_von: str, datum_bis: str) -> str:
    """Sucht Gesetzgebungsvorgänge im DIP des Bundestags nach Stichwort im Titel.

    Args:
        titel: Stichwort, z.B. "Mindestlohn".
        datum_von: Startdatum, Format YYYY-MM-DD.
        datum_bis: Enddatum, Format YYYY-MM-DD.
    """
    for name, wert in (("datum_von", datum_von), ("datum_bis", datum_bis)):
        if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", wert):
            raise ModelRetry(f"{name} muss YYYY-MM-DD sein, z.B. 2025-01-31")

    r = httpx.get(f"{DIP}/vorgang", timeout=15, headers={"Authorization": f"ApiKey {KEY}"},
                  params={"f.titel": titel, "f.datum.start": datum_von, "f.datum.end": datum_bis})
    if r.status_code == 401:
        raise ModelRetry("API-Key ungültig, Suche nicht möglich.")
    r.raise_for_status()

    docs = r.json()["documents"]
    if not docs:
        raise ModelRetry(f"Keine Treffer für '{titel}'. Versuch ein kürzeres Stichwort.")
    return "\n".join(f"{d['datum']} | {d['vorgangstyp']} | {d['titel']}" for d in docs[:10])



result = await agent.run("Was lief 2026 zum Mindestlohn?")
print(result.output)

Traceback (most recent call last):
  File "c:\Users\brand\anaconda3\envs\job_hf\Lib\site-packages\pydantic_ai\tool_manager.py", line 1056, in _raw_execute
    tool_result = await self.toolset.call_tool(
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<4 lines>...
    )
    ^
  File "c:\Users\brand\anaconda3\envs\job_hf\Lib\site-packages\pydantic_ai\toolsets\_tool_search.py", line 437, in call_tool
    return await self.wrapped.call_tool(name, tool_args, ctx, tool)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\brand\anaconda3\envs\job_hf\Lib\site-packages\pydantic_ai\toolsets\wrapper.py", line 70, in call_tool
    return await self.wrapped.call_tool(name, tool_args, ctx, tool)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\brand\anaconda3\envs\job_hf\Lib\site-packages\pydantic_ai\toolsets\combined.py", line 103, in call_tool
    return await tool.source_toolset.call_tool(
           ^^^^^^^^^^^^^^^^^^^^^

### Tagesschau Agent

In [2]:
@agent.tool_plain
def suche_nachrichten(suchtext: str) -> str:
    """Sucht aktuelle Nachrichten auf tagesschau.de.

    Args:
        suchtext: Suchbegriff, z.B. "Mindestlohn".
    """
    r = httpx.get("https://www.tagesschau.de/api2u/search/", timeout=15,
                  params={"searchText": suchtext, "pageSize": 100})
    r.raise_for_status()
    treffer = r.json().get("searchResults", [])
    if not treffer:
        raise ModelRetry(f"Keine Treffer für '{suchtext}'. Versuch einen anderen Begriff.")
    return "\n".join(f"{t.get('date', '?')[:10]} | {t.get('title', '?')}" for t in treffer)



NameError: name 'agent' is not defined

In [ ]:

result = await agent.run("Suche nach Neuigkeiten zu den GenZ Protesten in den Städten [Berlin, Frankfurt, Köln] am 31.10.2026.")


In [12]:
print(result)

AgentRunResult(output='Es scheint, dass es aktuell keine Nachrichten zum Thema "GenZ Demonstrationen 31.10.2026 Berlin München Hamburg" gibt. Möchten Sie, dass ich versuche, mit anderen Begriffen oder einem anderen Datum?\n\nAlternativ, können Sie mir mehr Informationen bereitstellen, um meine Suche zu optimieren?')
